In [ ]:
from dotenv import load_dotenv
load_dotenv()  # load API keys

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END, MessagesState

In [ ]:
# load the PDF document
loader = PyPDFLoader("../documents/evs_oil_price_shock.pdf")
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} pages")

In [ ]:
raw_docs[0].page_content

In [ ]:
# split documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)

chunks = splitter.split_documents(raw_docs)

print(f"Split into {len(chunks)} chunks")

In [ ]:
# initialize the embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [ ]:
# in-memory only; re-running this cell re-embeds from scratch
vectorstore = Chroma(
    collection_name="rag_base",
    embedding_function=embeddings,
)

vectorstore.add_documents(documents=chunks)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready")

In [ ]:
# graph state

class AgenticRAGState(MessagesState):
    
    query: str
    retrieved_docs: list[Document]
    context: str
    generation: str

In [ ]:
# initialize Groq (reads GROQ_API_KEY from .env)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0, max_retries=2)

In [ ]:
# retrieves relevant docs from the vector store

def retrieve(state: AgenticRAGState) -> dict:    
    
    docs = retriever.invoke(state["query"])
    
    context = "\n\n".join(doc.page_content for doc in docs)
    
    return {"retrieved_docs": docs, "context": context}

In [ ]:
# generates a response using the retrieved context

def generate(state: AgenticRAGState) -> dict:
    query = state["query"]
    context = state.get("context") or ""

    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "Answer the question using only the context below.\n\nContext:\n{context}"),
        ("human", "{query}"),
    ])

    response = (prompt_template | llm).invoke({"context": context, "query": query})

    return {"generation": response.content}

In [ ]:
# build the graph: START -> retrieve -> generate -> END
graph_builder = StateGraph(AgenticRAGState)

graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)

graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
graph

In [ ]:
test_query = "How will EVS impact oil demand in the next decade?"

result = graph.invoke({"query": test_query, "messages": []})

In [ ]:
print("=== GENERATED RESPONSE ===")
print(result["generation"])

In [ ]:
print("\n=== RETRIEVED DOCUMENTS ===")
for i, doc in enumerate(result["retrieved_docs"], 1):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "?")
    snippet = doc.page_content[:500].replace("\n", " ")
    print(f"[{i}] Source: {source} | Page: {page}")
    print(f"    {snippet}...")

In [ ]:
print(result["context"])